In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import dump
import warnings
warnings.filterwarnings('ignore')
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.compose import make_column_selector

from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler



from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import make_scorer, f1_score

In [2]:
fraud_train = pd.read_parquet('https://lab.cs307.org/fraud/data/fraud-train.parquet')
fraud_test = pd.read_parquet('https://lab.cs307.org/fraud/data/fraud-test.parquet')

X_train = fraud_train.drop('Fraud', axis = 1)
y_train = fraud_train['Fraud']

X_test = fraud_test.drop('Fraud', axis = 1)
y_test = fraud_test['Fraud']

In [ ]:
fc = fraud_train['Fraud'].value_counts()
fp = fraud_train['Fraud'].value_counts(normalize = True)
ft_b = pd.DataFrame({'count' : fc, 'fp' : fp})
ft_b

In [ ]:

genuine = fraud_train[fraud_train["Fraud"] == 0]
fraud = fraud_train[fraud_train["Fraud"] == 1]

print("None:")
print("Mean:", fraud_train["Amount"].mean())
print("Std:", fraud_train["Amount"].std())
print("Median:", fraud_train["Amount"].median())
print("Max:", fraud_train["Amount"].max())

print("\nGenuine transactions:")
print("Mean:", genuine["Amount"].mean())
print("Std:", genuine["Amount"].std())
print("Median:", genuine["Amount"].median())
print("Max:", genuine["Amount"].max())

print("\nFraudulent transactions:")
print("Mean:", fraud["Amount"].mean())
print("Std:", fraud["Amount"].std())
print("Median:", fraud["Amount"].median())
print("Max:", fraud["Amount"].max())

In [4]:
numeric_transformer = make_pipeline(SimpleImputer(strategy = 'median'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy = "most_frequent"), OneHotEncoder())

numeric_features = X_train.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include = ['object', 'category']).columns

preprocessor = make_column_transformer((numeric_transformer, numeric_features), 
                                       (categorical_transformer, categorical_features))
dec_tree = make_pipeline(preprocessor, DecisionTreeClassifier())

param_grid = {
    'decisiontreeclassifier__splitter': ['best'],  
    'decisiontreeclassifier__max_depth': [1, 2, 3, None],
    'decisiontreeclassifier__min_samples_leaf' : [1, 5],
    'decisiontreeclassifier__max_features' : [None, 'sqrt']
}

md = GridSearchCV(
    dec_tree, 
    param_grid, 
    cv = 3,
    scoring = 'recall'
)
md.fit(X_train, y_train)

p_score = precision_score(y_test, md.predict(X_test))
r_score = recall_score(y_test, md.predict(X_test))
print(p_score, r_score)

0.927536231884058 0.810126582278481


In [13]:
dump(md, 'fraud.joblib')

['fraud.joblib']

In [50]:
numeric_transformer = make_pipeline(SimpleImputer(strategy = 'median'), StandardScaler())
numeric_features = [
    'PC01', 'PC02', 'PC03', 'PC04', 'PC05', 'PC06', 'PC07', 'PC08', 'PC09', 'PC10',
    'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20',
    'PC21', 'PC22', 'PC23', 'PC24', 'PC25', 'PC26', 'PC27', 'PC28', 'Amount'
]


preprocessor = make_column_transformer((numeric_transformer, numeric_features))
decision_tree = make_pipeline(preprocessor, DecisionTreeClassifier(splitter = 'best'))

param_grid = { 
    'decisiontreeclassifier__min_samples_split' : [2, 4, 6, 8, 10],
    'decisiontreeclassifier__max_depth': [1, 2, None],
    'decisiontreeclassifier__min_samples_leaf' : [1, 5],
    #'decisiontreeclassifier__max_features' : [None, 'sqrt']
}

model = GridSearchCV(
    decision_tree, 
    param_grid, 
    cv = 3,
    scoring = 'recall'
)

model.fit(X_train, y_train)
p_score = precision_score(y_test, model.predict(X_test))
r_score = recall_score(y_test, model.predict(X_test))
print(p_score, r_score)

0.927536231884058 0.810126582278481
